# 🏋️ Numpy & Pandas 실습 — 벡터화 연산과 대용량 데이터 처리

📖 **자료 연계**: 「Numpy 벡터화 연산과 메모리 효율적 배열 처리」·「Pandas 대용량 데이터 처리 — chunksize와 dtype 최적화」

이 노트북은 두 자료에 나온 예제를 직접 실행하고, 숫자를 바꿔가며 결과가 어떻게
달라지는지 확인하는 실습용입니다. 정답을 맞히는 시험이 아니라 **"눈으로만 읽은 것을
손으로 직접 확인해보기"** 가 목적입니다.

**✅ 완료 기준**

**[Numpy]**
- [ ] 반복문과 벡터화 연산의 속도 차이를 직접 측정해봤다
- [ ] 브로드캐스팅으로 크기가 다른 배열끼리 연산해봤다
- [ ] dtype을 바꿔 메모리 사용량이 줄어드는 것을 확인했다
- [ ] view와 copy의 차이, in-place 연산의 이점을 코드로 확인했다
- [ ] 🔰 미션: 내가 만든 배열에 dtype 최적화를 적용해봤다

**[Pandas]**
- [ ] `memory_usage(deep=True)`로 컬럼별 실제 메모리를 확인했다
- [ ] 문자열 컬럼을 `category`로 바꿔 메모리 절감을 확인했다
- [ ] 값 가짓수가 많은 컬럼에는 `category`가 오히려 손해라는 것을 직접 확인했다
- [ ] `chunksize`로 파일을 나눠 읽고, 청크 결과를 잘못 합쳤을 때와 올바르게
      재집계했을 때의 차이를 확인했다

> ⚠️ **시작 전 확인**: numpy·pandas만 있으면 됩니다(둘 다 기본 포함인 경우가
> 대부분). 셀은 위에서부터 순서대로 실행하세요 — 뒤 셀이 앞 셀에서 만든 변수를
> 그대로 씁니다.

In [1]:
# ─── 환경 확인 ────────────────────────────────────────────────────────
import time
import numpy as np
import pandas as pd

print(f"✅ numpy {np.__version__} / pandas {pd.__version__} 준비 완료")

✅ numpy 2.5.3 / pandas 3.0.5 준비 완료


---
## 1부 · Numpy — 벡터화, 브로드캐스팅, 메모리

📖 **자료 연계**: 「Numpy 벡터화 연산과 메모리 효율적 배열 처리」

| 셀 | 단계 | 목표 |
|----|------|------|
| Step 1-① | 그대로 실행 | 반복문 vs 벡터화 속도를 직접 비교 |
| Step 1-② | 한 곳만 바꾸기 | 임계값(THRESHOLD)을 바꿔 마스킹 결과 확인 |
| Step 2 | 그대로 실행 | 브로드캐스팅으로 스칼라·배열 연산 적용 |
| Step 3-① | 그대로 실행 | dtype에 따른 메모리 크기 비교 |
| Step 3-② | 그대로 실행 | view와 copy의 차이 확인 |
| Step 3-③ | 그대로 실행 | in-place 연산으로 메모리 아끼기 |
| 🔰 미션 | 내 것에 적용 | 내가 만든 배열에 dtype 최적화 적용 |

In [2]:
# ─── Step 1-① 그대로 실행 — 반복문 vs 벡터화 속도 비교 ─────────────────
# 📖 자료 연계: 「Numpy ...」 모듈 1 · 왜 반복문이 아니라 벡터화인가

N = 2_000_000
data = list(range(N))          # 파이썬 리스트
arr = np.arange(N)             # Numpy 배열 (같은 값)

# ❌ 반복문 — 숫자 하나씩 파이썬 레벨에서 처리
t0 = time.perf_counter()
result_loop = [x * 2 + 1 for x in data]
t1 = time.perf_counter()

# ✅ 벡터화 — 배열 전체에 연산을 한 번에 적용
t2 = time.perf_counter()
result_vec = arr * 2 + 1
t3 = time.perf_counter()

loop_time, vec_time = t1 - t0, t3 - t2
print(f"반복문   : {loop_time:.4f}s")
print(f"벡터화   : {vec_time:.4f}s")
print(f"배수     : {loop_time / vec_time:.1f}배")
# 결과가 같은지도 확인 (속도만 다르고 값은 같아야 함)
print("결과 일치:", result_loop[:5] == list(result_vec[:5]))

반복문   : 0.0849s
벡터화   : 0.0081s
배수     : 10.5배
결과 일치: True


In [3]:
# ─── 참고 — 비교 연산도 벡터화됩니다 (임계값 넘는 값 찾기) ───────────────
np.random.seed(42)
response_ms = np.random.exponential(scale=50, size=1_000_000)  # 가짜 응답시간 100만 개
THRESHOLD = 200

# ❌ 반복문
t0 = time.perf_counter()
slow_count = sum(1 for x in response_ms if x > THRESHOLD)
t1 = time.perf_counter()

# ✅ 벡터화 — True/False 배열(마스크)이 나오고, True는 1로 취급되어 sum()으로 개수를 셈
t2 = time.perf_counter()
mask = response_ms > THRESHOLD
fast_count = mask.sum()
t3 = time.perf_counter()

print(f"임계값 {THRESHOLD}ms 초과 건수: 반복문={slow_count} / 벡터화={fast_count} (일치: {slow_count == fast_count})")
print(f"반복문 {t1 - t0:.4f}s vs 벡터화 {t3 - t2:.4f}s")

# 💡 핵심: for x in numpy_array: 는 벡터화가 아닙니다.
#          배열 단위 연산(>, *, np.where 등)으로 바꿔야 효과가 있습니다.

임계값 200ms 초과 건수: 반복문=18427 / 벡터화=18427 (일치: True)
반복문 0.0510s vs 벡터화 0.0013s


In [5]:
# ─── Step 1-② 한 곳만 바꾸기 — 임계값을 바꿔 마스킹 결과 확인 ───────────
# TODO(🔰): 아래 MY_THRESHOLD를 바꿔서 재실행해보세요.
#   힌트: 값을 낮추면 초과 건수가 늘고, 높이면 줄어듭니다.

MY_THRESHOLD = 300  # ← 여기를 바꿔보세요 (그대로도 실행됩니다)

my_mask = response_ms > MY_THRESHOLD
my_count = my_mask.sum()
ratio = my_count / len(response_ms) * 100

print(f"임계값 {MY_THRESHOLD}ms 초과 건수: {my_count:,}건 (전체의 {ratio:.2f}%)")
print("mask dtype:", my_mask.dtype, "→ True/False로 이루어진 배열 자체도 하나의 Numpy 배열입니다.")

임계값 300ms 초과 건수: 2,479건 (전체의 0.25%)
mask dtype: bool → True/False로 이루어진 배열 자체도 하나의 Numpy 배열입니다.


### 브로드캐스팅 — 크기가 다른 배열끼리 연산하기

📖 **자료 연계**: 「Numpy 벡터화 연산과 메모리 효율적 배열 처리」모듈 2 · 브로드캐스팅

In [6]:
# ─── Step 2 그대로 실행 — 브로드캐스팅 데모 ─────────────────────────────
durations_ms = np.array([120, 450, 80, 900, 60])   # 요청 5개의 처리 시간
threshold = 200                                     # 숫자 하나(스칼라)

# 스칼라 하나가 배열 크기(5)에 맞춰 "복제된 것처럼" 적용됨
over_threshold = durations_ms > threshold
print("초과 여부:", over_threshold)

# 초 단위로 변환 — 역시 스칼라 나눗셈이 배열 전체에 적용
durations_sec = durations_ms / 1000
print("초 단위  :", durations_sec)

초과 여부: [False  True False  True False]
초 단위  : [0.12 0.45 0.08 0.9  0.06]


In [7]:
# ─── 🔰 한 곳만 바꾸기 — 내 데이터로 브로드캐스팅 적용 ───────────────────
# TODO(🔰): MY_DURATIONS와 MY_THRESHOLD_MS를 바꿔보세요.

MY_DURATIONS = np.array([300, 50, 700, 20, 150, 999])  # ← 내 요청 처리시간(ms)으로 교체
MY_THRESHOLD_MS = 250                                    # ← 내 기준값으로 교체

print("초과 여부 :", MY_DURATIONS > MY_THRESHOLD_MS)
print("초 단위   :", MY_DURATIONS / 1000)
print(f"초과 건수 : {(MY_DURATIONS > MY_THRESHOLD_MS).sum()} / {len(MY_DURATIONS)}건")

초과 여부 : [ True False  True False False  True]
초 단위   : [0.3   0.05  0.7   0.02  0.15  0.999]
초과 건수 : 3 / 6건


### dtype과 메모리 — 상자 크기와 view/copy

📖 **자료 연계**: 「Numpy 벡터화 연산과 메모리 효율적 배열 처리」모듈 3 · 메모리 효율적 배열 처리

In [8]:
# ─── Step 3-① 그대로 실행 — dtype별 메모리 크기 비교 ────────────────────
status_codes = np.array([200, 404, 500, 200, 301] * 200_000)  # HTTP 상태코드 100만 개

print("dtype (기본값)  :", status_codes.dtype)
print("메모리(bytes)   :", status_codes.nbytes, f"(약 {status_codes.nbytes / 1e6:.1f}MB)")

# 상태코드는 0~599 범위 → int16(약 -32768~32767)이면 충분
small = status_codes.astype(np.int16)   # astype은 항상 복사본을 만듦
print("dtype (int16)   :", small.dtype)
print("메모리(bytes)   :", small.nbytes, f"(약 {small.nbytes / 1e6:.1f}MB, 원래의 {small.nbytes / status_codes.nbytes:.0%})")

dtype (기본값)  : int64
메모리(bytes)   : 8000000 (약 8.0MB)
dtype (int16)   : int16
메모리(bytes)   : 2000000 (약 2.0MB, 원래의 25%)


In [9]:
# ─── Step 3-② 그대로 실행 — view와 copy의 차이 ──────────────────────────
original = np.array([1, 2, 3, 4, 5])

sliced = original[1:4]      # ← view. 새 메모리를 쓰지 않고 원본을 "가리키기만" 함
sliced[0] = 999
print("sliced 수정 후 original:", original, "← view라서 원본도 바뀜")

original2 = np.array([1, 2, 3, 4, 5])
copied = original2[1:4].copy()   # ← 명시적으로 복사본을 만들면 원본과 독립됨
copied[0] = -1
print("copied 수정 후 original2:", original2, "← copy라서 원본은 안 바뀜")

# ⚠️ 흔한 실수: 슬라이싱 결과를 "당연히 복사본"이라 생각하고 수정했다가
#              원본까지 같이 바뀌는 경우입니다. 원본 보존이 필요하면 .copy()를 명시하세요.

sliced 수정 후 original: [  1 999   3   4   5] ← view라서 원본도 바뀜
copied 수정 후 original2: [1 2 3 4 5] ← copy라서 원본은 안 바뀜


In [10]:
# ─── Step 3-③ 그대로 실행 — in-place 연산으로 메모리 아끼기 ─────────────
big = np.zeros(5_000_000)
before_id = id(big)

# ❌ 매번 새 배열을 만들어 메모리를 두 배로 순간 사용
big_new = big + 1
print("big + 1  → 새 배열인가?", id(big_new) != before_id)

# ✅ 기존 메모리 자리에서 바로 값을 바꿈 — 새 배열을 만들지 않음
big += 1
print("big += 1 → 같은 배열인가?", id(big) == before_id)
print("값 확인 (앞 3개):", big[:3])

big + 1  → 새 배열인가? True
big += 1 → 같은 배열인가? True
값 확인 (앞 3개): [1. 1. 1.]


In [11]:
# ─── 🔰 Numpy 미션 — 내 배열에 dtype 최적화 적용해보기 ──────────────────
# TODO(🔰): MY_ARRAY를 원하는 정수 배열로 바꾸세요. (그대로도 실행됩니다)
#   순서: ① 최댓값·최솟값 확인 → ② 값 범위에 맞는 dtype 선택 → ③ 메모리 절감 확인
#   ⚠️ dtype을 너무 좁게 잡으면 오버플로가 나서 값이 깨집니다 — 범위 확인이 먼저입니다.

MY_ARRAY = np.random.randint(0, 333, size=333333333)  # ← 내 정수 데이터로 교체

print("최솟값/최댓값:", MY_ARRAY.min(), "/", MY_ARRAY.max())
print("원래 dtype   :", MY_ARRAY.dtype, "→", MY_ARRAY.nbytes, "bytes")

# 값 범위가 int16 범위(-32768~32767) 안에 들어오는지 확인 후 다운캐스팅
assert MY_ARRAY.min() >= -32768 and MY_ARRAY.max() <= 32767, "int16 범위를 벗어납니다 — dtype을 다시 선택하세요"
optimized = MY_ARRAY.astype(np.int16)
saved = 1 - optimized.nbytes / MY_ARRAY.nbytes

print(f"최적화 dtype : {optimized.dtype} → {optimized.nbytes:,} bytes")
print(f"메모리 절감량: {saved:.0%}")

최솟값/최댓값: 0 / 332
원래 dtype   : int32 → 1333333332 bytes
최적화 dtype : int16 → 666,666,666 bytes
메모리 절감량: 50%


---
## 2부 · Pandas — 메모리 최적화와 chunksize

📖 **자료 연계**: 「Pandas 대용량 데이터 처리 — chunksize와 dtype 최적화」

| 셀 | 단계 | 목표 |
|----|------|------|
| Step 4-① | 그대로 실행 | `memory_usage(deep=True)`로 컬럼별 메모리 확인 |
| Step 4-② | 한 곳만 바꾸기 | 정수 다운캐스팅 + `category` 변환으로 메모리 절감 |
| 🔰 미션 | 내 것에 적용 | 값 가짓수가 많은 컬럼엔 `category`가 손해임을 확인 |
| Step 5-① | 그대로 실행 | `chunksize`로 큰 파일 나눠 읽기 + 잘못된 합산 재현 |
| Step 5-② | 한 곳만 바꾸기 | 올바른 재집계로 수정, 정답과 대조 |
| 🔰 미션 | 내 것에 적용 | chunksize를 바꿔가며 처리 시간 비교 |

In [12]:
# ─── Step 4-① 그대로 실행 — Pandas가 메모리를 많이 먹는 이유 ────────────
np.random.seed(42)
n = 1_000_000
df = pd.DataFrame({
    "status_code": np.random.choice([200, 301, 404, 500], size=n),   # 값 종류 4개뿐
    "level": np.random.choice(["INFO", "WARN", "ERROR"], size=n),    # 값 종류 3개뿐
    "response_ms": np.random.exponential(scale=50, size=n),
})

print(df.dtypes)
print()
mem_before = df.memory_usage(deep=True)
print(mem_before)
print(f"\n합계: {mem_before.sum() / 1e6:.1f}MB")
# ⚠️ 흔한 실수: df.info()만 보고 넘어가면 문자열 컬럼의 실제 메모리를 과소평가합니다.
#              memory_usage(deep=True) 또는 df.info(memory_usage="deep")을 써야 합니다.

status_code      int64
level              str
response_ms    float64
dtype: object

Index               132
status_code     8000000
level          53333646
response_ms     8000000
dtype: int64

합계: 69.3MB


In [13]:
# ─── Step 4-② 한 곳만 바꾸기 — dtype 최적화 적용 ────────────────────────
# TODO(🔰): 다른 정수 컬럼이 있다면 astype 줄을 하나 더 추가해보세요.

df_opt = df.copy()
df_opt["status_code"] = df_opt["status_code"].astype("int16")   # ① 정수 다운캐스팅
df_opt["level"] = df_opt["level"].astype("category")            # ② 값 가짓수 적은 문자열 → category

mem_after = df_opt.memory_usage(deep=True)
compare = pd.DataFrame({"최적화 전": mem_before, "최적화 후": mem_after})
compare["절감률"] = 1 - compare["최적화 후"] / compare["최적화 전"]
print(compare)
print(f"\n전체 절감률: {1 - mem_after.sum() / mem_before.sum():.0%}")

                최적화 전    최적화 후       절감률
Index             132      132  0.000000
status_code   8000000  2000000  0.750000
level        53333646  1000160  0.981247
response_ms   8000000  8000000  0.000000

전체 절감률: 84%


In [14]:
# ─── 🔰 Pandas 미션 — category가 오히려 손해인 경우 직접 확인 ───────────
# 값 가짓수가 아주 많은 컬럼(예: 요청 ID)에 category를 적용하면 어떻게 될까요?

df_id = df.copy()
df_id["request_id"] = [f"req-{i}" for i in range(len(df_id))]  # 거의 전부 다른 값

before = df_id["request_id"].memory_usage(deep=True)
after = df_id["request_id"].astype("category").memory_usage(deep=True)

print("request_id 값 가짓수(nunique):", df_id["request_id"].nunique(), "/ 전체 행:", len(df_id))
print(f"object   : {before / 1e6:.1f}MB")
print(f"category : {after / 1e6:.1f}MB")
print("→ category가 더 작은가?", after < before)
# 💡 핵심: category 적용 전 반드시 nunique()로 값의 가짓수를 먼저 확인하세요.

request_id 값 가짓수(nunique): 1000000 / 전체 행: 1000000
object   : 58.9MB
category : 62.9MB
→ category가 더 작은가? False


### chunksize — 파일을 나눠서 읽기

📖 **자료 연계**: 「Pandas 대용량 데이터 처리 — chunksize와 dtype 최적화」 모듈 3 · chunksize

In [15]:
# ─── Step 5-① 그대로 실행 — 샘플 로그 CSV 만들고 chunksize로 읽기 ───────
import os

os.makedirs("data", exist_ok=True)
np.random.seed(42)

n_rows = 240_000
hosts = np.random.choice(["host-a", "host-b", "host-c"], size=n_rows)  # 3대뿐 → 여러 청크에 걸쳐 등장
sample_log = pd.DataFrame({
    "host": hosts,
    "level": np.random.choice(["INFO", "WARN", "ERROR"], size=n_rows, p=[0.7, 0.2, 0.1]),
    "response_ms": np.random.exponential(scale=50, size=n_rows),
})
sample_log.to_csv("data/sample_log.csv", index=False)
print(f"data/sample_log.csv 생성 완료 — {n_rows:,}행")

# chunksize를 주면 DataFrame이 아니라 "청크를 하나씩 내어주는 이터레이터"가 반환됨
CHUNKSIZE = 50_000
partial_sums = []
for chunk in pd.read_csv("data/sample_log.csv", chunksize=CHUNKSIZE):
    partial_sums.append(chunk.groupby("host")["response_ms"].sum())

# ❌ 청크별 groupby 결과를 그냥 이어붙이기만 함 — 같은 host가 여러 청크에 걸쳐 나타나 행이 중복됨
wrong = pd.concat(partial_sums)
print("\n[잘못된 방식] concat만 한 결과 (행 개수가 host 3개보다 많음):")
print(wrong)

data/sample_log.csv 생성 완료 — 240,000행

[잘못된 방식] concat만 한 결과 (행 개수가 host 3개보다 많음):
host
host-a    838159.706401
host-b    841609.449197
host-c    829979.853481
host-a    828998.478118
host-b    841139.866694
host-c    817740.111370
host-a    842749.855471
host-b    825052.058330
host-c    845363.904927
host-a    838713.741577
host-b    838855.828463
host-c    826354.092696
host-a    667354.958174
host-b    674387.460458
host-c    663885.042534
Name: response_ms, dtype: float64


In [16]:
# ─── Step 5-② 한 곳만 바꾸기 — 올바른 재집계로 수정 ─────────────────────
# ✅ 이어붙인 뒤 같은 키(host)로 다시 한 번 집계
right = pd.concat(partial_sums).groupby(level=0).sum()

# 정답: 파일 전체를 한 번에 읽어서 집계한 결과
answer = pd.read_csv("data/sample_log.csv").groupby("host")["response_ms"].sum()

print("[올바른 방식] concat 후 재집계:")
print(right)
print("\n[정답] 전체를 한 번에 읽어 집계:")
print(answer)

# 두 결과가 (부동소수점 오차 범위 내에서) 같은지 확인
pd.testing.assert_series_equal(right.sort_index(), answer.sort_index())
print("\n✅ 재집계 결과가 정답과 일치합니다.")
# ⚠️ 흔한 실수: groupby 결과를 concat만 하고 끝내면 에러 없이 조용히 틀린 숫자가 나옵니다.

[올바른 방식] concat 후 재집계:
host
host-a    4.015977e+06
host-b    4.021045e+06
host-c    3.983323e+06
Name: response_ms, dtype: float64

[정답] 전체를 한 번에 읽어 집계:
host
host-a    4.015977e+06
host-b    4.021045e+06
host-c    3.983323e+06
Name: response_ms, dtype: float64

✅ 재집계 결과가 정답과 일치합니다.


In [17]:
# ─── 🔰 종합 미션 — chunksize를 바꿔가며 비교 + dtype 최적화 함께 적용 ───
# TODO(🔰): CHUNK_SIZES 목록을 바꿔가며 처리 시간이 어떻게 달라지는지 확인하세요.
#   힌트: 너무 작으면 나눠 읽는 오버헤드가 커져 오히려 느려질 수 있습니다.

CHUNK_SIZES = [10_000, 50_000, 120_000]

for cs in CHUNK_SIZES:
    t0 = time.perf_counter()
    total_errors = 0
    for chunk in pd.read_csv(
        "data/sample_log.csv",
        chunksize=cs,
        dtype={"host": "category", "level": "category"},   # 읽을 때부터 dtype 지정
    ):
        total_errors += (chunk["level"] == "ERROR").sum()
    elapsed = time.perf_counter() - t0
    print(f"chunksize={cs:>7,} | ERROR 총 {total_errors:>6,}건 | {elapsed:.3f}s")

# 💡 핵심: 처음부터 dtype을 지정해서 읽으면(dtype=...), 기본값으로 읽은 뒤
#          astype()으로 바꾸는 것보다 애초에 메모리를 덜 씁니다.

chunksize= 10,000 | ERROR 총 23,912건 | 0.149s
chunksize= 50,000 | ERROR 총 23,912건 | 0.088s
chunksize=120,000 | ERROR 총 23,912건 | 0.057s


In [ ]:
#titanic 데이터로 내가 직접 해보기 1
CHUNKSIZE = 200
partial_sums = []
for chunk in pd.read_csv("data/Titanic_train.csv", chunksize=CHUNKSIZE):
    partial_sums.append(chunk.groupby("Pclass")["Fare"].sum())
right = pd.concat(partial_sums).groupby(level=0).sum()
print("class별로 Fare 합 정리:")
print(right)

class별로 Fare 합 정리:
Pclass
1    18177.4125
2     3801.8417
3     6714.6951
Name: Fare, dtype: float64


In [ ]:
#titanic 데이터로 내가 직접 해보기 2
CHUNKSIZE = 200
partial_sums = []
for chunk in pd.read_csv("data/Titanic_train.csv", chunksize=CHUNKSIZE):
    partial_sums.append(chunk.groupby("Survived")["Fare"].mean())
right = pd.concat(partial_sums).groupby(level=0).sum()
print("Survived별로 Fare 평균:")
print(right)

Survived별로 Fare 평균:
Survived
0    107.142315
1    235.014521
Name: Fare, dtype: float64


---
## 📬 자가 체크

**[Numpy]**
- [ ] 반복문과 벡터화 연산의 속도 차이를 직접 측정해봤다
- [ ] 브로드캐스팅으로 크기가 다른 배열끼리 연산해봤다
- [ ] dtype을 바꿔 메모리 사용량이 줄어드는 것을 확인했다
- [ ] view와 copy의 차이, in-place 연산의 이점을 코드로 확인했다
- [ ] 🔰 미션: 내가 만든 배열에 dtype 최적화를 적용해봤다

**[Pandas]**
- [ ] `memory_usage(deep=True)`로 컬럼별 실제 메모리를 확인했다
- [ ] 문자열 컬럼을 `category`로 바꿔 메모리 절감을 확인했다
- [ ] 값 가짓수가 많은 컬럼에는 `category`가 오히려 손해라는 것을 직접 확인했다
- [ ] `chunksize`로 파일을 나눠 읽고, 잘못된 합산과 올바른 재집계의 차이를 확인했다

---
### ➡️ 다음 연결

오늘 다룬 두 축 — **계산을 빠르게(벡터화·브로드캐스팅)**, **메모리를 아껴서
(dtype·category·view)**, **한 번에 다 올리지 않고(chunksize)** — 는 모두
"메모리·계산 효율"에 관한 것입니다. "여러 코어에 일을 나누는 것"(병렬 처리)은
다음에 별도로 다루는 다른 층위의 최적화입니다 — 둘을 헷갈리지 않도록
기억해두면 이후 내용이 더 가볍게 느껴질 것입니다.